In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd

INPUT, WORK = Path('/kaggle/input'), Path('/kaggle/working')
def newest(name):
    hits=list(INPUT.rglob(name))
    if not hits: raise FileNotFoundError(f'Missing {name}; attach the saved Day 9.19 and Day 9.16 outputs.')
    return max(hits,key=lambda p:(p.stat().st_mtime,str(p)))

NEW_CASE=newest('day9_19_case_diagnostics.csv')
NEW_PARAM=newest('day9_19_per_parameter_diagnostics.csv')
NEW_INTERVAL=newest('day9_19_posterior_intervals.csv')
OLD_PARAM=newest('day9_16_per_parameter_diagnostics.csv')
OLD_CASE=newest('day9_16_case_diagnostics.csv')
cases=pd.read_csv(NEW_CASE); params=pd.read_csv(NEW_PARAM); intervals=pd.read_csv(NEW_INTERVAL)
old_params=pd.read_csv(OLD_PARAM); old_cases=pd.read_csv(OLD_CASE)
for df in (cases,params,intervals,old_params,old_cases):
    if 'simulation_row' in df: df['simulation_row']=df['simulation_row'].astype(int)

# Report the familiar convergence criteria separately from tree-depth efficiency warnings.
# R-hat <= 1.05, ESS >= 100, and zero divergences are the core screening thresholds here.
rows=[]
for _,r in cases.iterrows():
    rid=int(r.simulation_row); sub=params[params.simulation_row.eq(rid)]
    rh=pd.to_numeric(sub.r_hat,errors='coerce').to_numpy(float)
    ess=pd.to_numeric(sub.effective_sample_size,errors='coerce').to_numpy(float)
    finite=np.isfinite(rh)&np.isfinite(ess)
    max_rhat=float(np.nanmax(rh)) if np.isfinite(rh).any() else np.nan
    min_ess=float(np.nanmin(ess)) if np.isfinite(ess).any() else np.nan
    rhat_ok=bool(finite.all() and max_rhat<=1.05)
    ess_ok=bool(finite.all() and min_ess>=100)
    div_ok=int(r.divergences)==0
    core='pass' if rhat_ok and ess_ok and div_ok else 'fail'
    tree_fraction=float(r.fraction_at_max_tree_depth)
    tree_warning=tree_fraction>=0.05
    old_noise=old_params[(old_params.simulation_row.eq(rid)) & (old_params.parameter.eq('log_noise'))]
    old_noise_ess=float(old_noise.iloc[0].effective_sample_size) if len(old_noise)==1 else np.nan
    new_noise= sub[sub.parameter.eq('log_noise')]
    new_noise_ess=float(new_noise.iloc[0].effective_sample_size) if len(new_noise)==1 else np.nan
    rows.append({'simulation_row':rid,'core_convergence_screen':core,'max_r_hat':max_rhat,
        'r_hat_threshold':1.05,'minimum_ess':min_ess,'ess_threshold':100,
        'divergences':int(r.divergences),'tree_depth_warning':tree_warning,
        'fraction_at_max_tree_depth':tree_fraction,'day9_16_log_noise_ess':old_noise_ess,
        'day9_19_log_noise_ess':new_noise_ess,
        'interpretation':('Unresolved severe chain non-mixing; fixing duration and marginalizing noise did not solve it.' if rid==87483 else
          'Core diagnostics pass; high maximum-tree-depth use is an efficiency warning that should be reported.')})
score=pd.DataFrame(rows).sort_values('simulation_row')

coverage_rows=[]
if len(intervals):
    intervals['truth_inside_90pct_interval']=intervals.truth_inside_90pct_interval.astype(str).str.lower().map({'true':True,'false':False,'1':True,'0':False})
    for rid,g in intervals.groupby('simulation_row'):
        # log_noise and noise_sigma are the same physical quantity; count noise_sigma once.
        g=g[~g.parameter.eq('log_noise')].copy()
        n=int(g.truth_inside_90pct_interval.notna().sum())
        hit=int(g.truth_inside_90pct_interval.fillna(False).sum())
        coverage_rows.append({'simulation_row':int(rid),'truth_inside_count':hit,
            'estimated_parameter_intervals':n,'coverage_scope':'conditional synthetic case check; not calibration'})
coverage=pd.DataFrame(coverage_rows)
OUT=WORK/('day9-20-marginalized-nuts-audit-'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
OUT.mkdir(parents=True,exist_ok=False)
score.to_csv(OUT/'day9_20_case_audit.csv',index=False)
params.to_csv(OUT/'day9_20_per_parameter_diagnostics.csv',index=False)
coverage.to_csv(OUT/'day9_20_conditional_truth_checks.csv',index=False)

n=len(score); core_pass=int(score.core_convergence_screen.eq('pass').sum())
report={'created_utc':datetime.now(timezone.utc).isoformat(),
    'sources':{'day9_19_cases':str(NEW_CASE),'day9_19_parameters':str(NEW_PARAM),
      'day9_19_intervals':str(NEW_INTERVAL),'day9_16_parameters':str(OLD_PARAM),'day9_16_cases':str(OLD_CASE)},
    'cases':n,'core_screen_pass':core_pass,'core_screen_fail':n-core_pass,
    'core_thresholds':{'max_r_hat':1.05,'min_ess':100,'divergences':0},
    'tree_depth_note':'Fraction at maximum tree depth is reported as an efficiency warning separately; the 5% cap used by Day 9.19 was a conservative project-specific rule, not part of the core thresholds above.',
    'limits':['Both cases condition on a fixed period; row 87483 also conditions on a fixed duration.',
      'Only two synthetic cases were evaluated; interval checks are not a coverage calibration study.',
      'Passing sampler diagnostics does not prove the physical model is correct or valid on real stars.'],
    'outputs':['day9_20_case_audit.csv','day9_20_per_parameter_diagnostics.csv','day9_20_conditional_truth_checks.csv','day9_20_audit_summary.json','day9_20_audit_report.md']}
(OUT/'day9_20_audit_summary.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
md=['# Day 9.20 — Marginalized-noise NUTS audit','',
    f'Core convergence screen: **{core_pass}/{n} pass** using R-hat ≤ 1.05, ESS ≥ 100, and zero divergences. Tree-depth use is reported separately.', '',
    '| Row | Core screen | Max R-hat | Min ESS | Divergences | At max depth | log_noise ESS Day 9.16 → 9.19 |',
    '|---:|---|---:|---:|---:|---:|---:|']
for _,r in score.iterrows():
    md.append(f"| {int(r.simulation_row)} | {r.core_convergence_screen} | {r.max_r_hat:.4g} | {r.minimum_ess:.4g} | {r.divergences} | {r.fraction_at_max_tree_depth:.1%} | {r.day9_16_log_noise_ess:.1f} → {r.day9_19_log_noise_ess:.1f} |")
md += ['', '## Interpretation','',
    '- **Row 4923:** the core R-hat/ESS/divergence screen passes after exact noise marginalization. Its maximum-tree-depth use remains high (about 71%), so report that efficiency warning; Day 9.19’s stricter `needs_review` status came from adding a 5% tree-depth cap.',
    '- **Row 87483:** remains unresolved. Its chains still fail badly even after noise marginalization, with R-hat far above threshold and ESS near 1.',
    '- This is conditional synthetic recovery for two cases, not calibration and not real-star validation.', '',
    '## Recommended next step','',
    'Use the row 4923 sampler configuration as a candidate for a broader, predeclared synthetic validation set. Keep row 87483 in the failure/triage set; do not keep spending GPU time repeating its current fit. Measure recovery and interval coverage across multiple independent cases before making a broad Bayesian-system claim.', '']
(OUT/'day9_20_audit_report.md').write_text('\n'.join(md),encoding='utf-8')
print(score.to_string(index=False))
print('\nConditional truth checks (not calibration):\n',coverage.to_string(index=False))
print('\nSaved audit outputs:',OUT)
print(json.dumps(report,indent=2))